# E04 — Thermal and cooling

| | |
|---|---|
| **Question** | Can cooling, outdoor-temperature dependence, room temperature and waste heat be represented in Vessim, and with what fidelity loss compared with the edge study? |
| **Hypothesis** | H2 (Vessim has no thermal model; cooling can only be approximated as an extra electrical consumer) |
| **Branch** | `exp/e04-thermal-cooling` |
| **Sites** | Edge study S1.2 (2025-04-01, cooling chain without heat recovery) and S2.2 (2025-04-04, waste heat to a warm district-heating network), one day each, 15-min IT power and outdoor temperature from the edge runs |

**Variants**

| Variant | How it is built in Vessim |
|---|---|
| Constant overhead (0.285 × IT) | built-in `Consumer` with a precomputed trace — the planning assumption in the edge study's controller and scheduler |
| Linear fit to edge run | `CoolingProxy` actor from E03 (slope · IT + intercept) |
| Ported edge cooling chain | custom `ChainCooling` actor: CRAH fan, trim chiller with a COP from the outdoor temperature, two pumps — equations of the edge study's `cooling_crah.py`, `cooling_chiller.py`, `pump_dc.py`, `pump_chiller_evap.py` |
| Room thermal node | custom `RoomThermal` actor: one room-air temperature state (thermal capacity), proportional cooling control, precooling and curtailment windows |
| Waste-heat recovery | custom `HeatRecovery` actor (heat pump to district heating) + `DeliveredHeat` metric; compared with the edge run of S2.2 |

Vessim's own source contains no thermal term at all (no heat, temperature or cooling). Every thermal quantity below is
user code; Vessim only sees the electrical power each actor returns.

## 1. Setup

In [1]:
import importlib.metadata as md_
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

from vessim_eval import RESULTS_DIR
from vessim_eval import plotting as vp
from vessim_eval.dispatch import logic_lines, price_from_reference
from vessim_eval.edge_reference import load_reference, scenario_config
from vessim_eval.thermal import (EDGE_COOLING_OVERHEAD, ChainCooling, CoolingChain, DeliveredHeat,
                                 HeatPumpParams, HeatRecovery, PowerProbe, RoomThermal, Window,
                                 cooling_agreement, curtailment_kpis, energy_kwh, heat_demand_kw,
                                 heat_kpis, run_thermal)
from vessim_eval.workload import CoolingProxy

print("vessim", md_.version("vessim"))

vessim 0.16.0


In [2]:
# ---- Parameters -------------------------------------------------------------
STEP_S = 900                                  # edge physical step
TEMP_OFFSETS_K = [0, 5, 10, 15, 20, 25]       # uniform shifts of the S1.2 outdoor temperature
H = 3600
ROOM = dict(capacity_mj_k=240.0,              # ≈ 1 K/min rise after a full cooling stop at 4 MW
            t_set_c=24.0, t_max_c=27.0,       # server-inlet set point and limit (ASHRAE recommended ≤ 27 °C)
            tau_s=900.0)                      # proportional-control time constant
EVENT = Window(5 * H, int(5.5 * H), max_removal_frac=0.5)   # 05:00–05:30, highest price of the day
PRECOOL = Window(4 * H, 5 * H, setpoint_c=20.0)             # 04:00–05:00
ROOM_STEPS_S = {"1 min": 60, "5 min": 300, "15 min": 900}
CAPACITY_SWEEP_MJ_K = [60, 240, 960]
OUT = RESULTS_DIR / "E04"
OUT.mkdir(parents=True, exist_ok=True)

s12 = load_reference("e04_edge_s1_2_cooling_15min.csv")    # edge cooling chain, S1.2
site = load_reference("e01_edge_s1_2_15min.csv")           # building, PV, price (S1.2)
price = price_from_reference(site)                         # €/MWh
s22 = load_reference("e04_edge_s2_2_heat_15min.csv")       # edge heat recovery, S2.2

# S2.2 cooling-chain overrides (config) and heat-pump calibration from the edge run
hn = scenario_config("s2_2_waste_heat_warm")["data_centers"][0]["heat_network_config"]
chain22 = CoolingChain(water_dt_k=hn["cooling"]["crah"]["DeltaT_water_target_k"])
hp_on = s22.heat_pump_kw > 1
hp = HeatPumpParams(capacity_kw=hn["cooling"]["hp"]["Q_cond_nom_W"] / 1e3,
                    t_sink_c=hn["cooling"]["hp"]["default_T_cond_out_set_C"],
                    t_source_c=((s22.hp_source_in_c + s22.hp_source_out_c) / 2)[hp_on].mean(),
                    delivery_cap_kw=s22.dh_hx_heat_kw.max())
chain = CoolingChain()
pd.Series({
    "S1.2 outdoor temperature (°C)": f"{s12.ambient_temp_c.min():.1f} to {s12.ambient_temp_c.max():.1f}",
    "S1.2 IT power (kW)": f"{s12.it_kw.min():,.0f} to {s12.it_kw.max():,.0f}",
    "CRAH heat capacity at design air flow (kW)": f"{chain.capacity_kw:,.0f}",
    "Chiller COP on the S1.2 day": f"{chain.cop(s12.ambient_temp_c).min():.2f} to {chain.cop(s12.ambient_temp_c).max():.2f}",
    "S2.2 outdoor temperature (°C)": f"{s22.ambient_temp_c.min():.1f} to {s22.ambient_temp_c.max():.1f}",
    "S2.2 heat pump: source / sink (°C), COP": f"{hp.t_source_c:.1f} / {hp.t_sink_c:.0f}, {hp.cop:.2f} (edge mean {s22.heat_pump_cop[hp_on].mean():.2f})",
    "S2.2 heat to network, edge maximum (kW)": f"{hp.delivery_cap_kw:,.1f}",
}, name="Value").to_frame()

,Value
S1.2 outdoor temperature (°C),1.7 to 13.4
S1.2 IT power (kW),"1,873 to 4,939"
CRAH heat capacity at design air flow (kW),"5,000"
Chiller COP on the S1.2 day,4.50 to 4.50
S2.2 outdoor temperature (°C),4.5 to 18.6
"S2.2 heat pump: source / sink (°C), COP","29.1 / 55, 6.09 (edge mean 6.12)"
"S2.2 heat to network, edge maximum (kW)","2,123.5"


The edge study connects no room-air model to its CRAH, so the CRAH works with a constant return-air temperature of
30 °C, and the chiller holds the chilled water at its 15 °C set point. In heat mode `none` (S1.2) the whole chain is
therefore an algebraic function of IT heat and outdoor temperature, evaluated once per 15-min physical step with the IT
power at the start of the step.

## 2. Run

In [3]:
import time
t_start = time.perf_counter()

# A. Cooling models on S1.2 (IT at the start of each step, as the edge chain uses it)
slope, intercept = np.polyfit(s12.it_kw, s12.cooling_kw, 1)
MODELS = {"Constant overhead (0.285 × IT)": dict(model="constant_overhead"),
          "Linear fit to edge run": dict(model="linear_fit", linear=(slope, intercept)),
          "Ported edge cooling chain": dict(model="chain")}
models = {n: run_thermal(s12.it_kw, s12.ambient_temp_c, step_s=STEP_S, **kw) for n, kw in MODELS.items()}
chain_on_mean_it = run_thermal(s12.it_mean_kw, s12.ambient_temp_c, "chain", step_s=STEP_S)

# B. Outdoor-temperature sensitivity (uniform shifts of the S1.2 day)
shifted = {dt: {n: run_thermal(s12.it_kw, s12.ambient_temp_c + dt, step_s=STEP_S, **MODELS[n])
                for n in ("Constant overhead (0.285 × IT)", "Ported edge cooling chain")}
           for dt in TEMP_OFFSETS_K}

# C. Room node: curtailment event, with and without precooling, three step sizes
room_base = run_thermal(s12.it_kw, s12.ambient_temp_c, "room", step_s=STEP_S, room=ROOM)
room = {(case, lbl): run_thermal(s12.it_kw, s12.ambient_temp_c, "room", step_s=s,
                                 room={**ROOM, "windows": w})
        for case, w in (("Curtail", [EVENT]), ("Precool + curtail", [PRECOOL, EVENT]))
        for lbl, s in ROOM_STEPS_S.items()}
room_capacity = {c: run_thermal(s12.it_kw, s12.ambient_temp_c, "room", step_s=60,
                                room={**ROOM, "capacity_mj_k": c, "windows": [PRECOOL, EVENT]})
                 for c in CAPACITY_SWEEP_MJ_K}

# D. Waste-heat recovery on S2.2
heat = {"Vessim heat-recovery actor": run_thermal(
            s22.it_kw, s22.ambient_temp_c, "heat_recovery", step_s=STEP_S, chain=chain22,
            heat_demand=s22.heat_demand_kw, hp=hp),
        "Vessim actor, no exchanger limit": run_thermal(
            s22.it_kw, s22.ambient_temp_c, "heat_recovery", step_s=STEP_S, chain=chain22,
            heat_demand=s22.heat_demand_kw, hp=replace(hp, delivery_cap_kw=np.inf))}

n_runs = 3 + 1 + 2 * len(TEMP_OFFSETS_K) + 1 + len(room) + len(room_capacity) + len(heat)
print(f"{n_runs} Vessim runs in {time.perf_counter() - t_start:.2f} s")
print("one day at 15 min:", f"{models['Ported edge cooling chain']['wall_s'] * 1e3:.0f} ms;",
      "at 1 min:", f"{room[('Curtail', '1 min')]['wall_s'] * 1e3:.0f} ms")

28 Vessim runs in 0.75 s
one day at 15 min: 5 ms; at 1 min: 81 ms


## 3. Results

### 3.1 Cooling models vs the edge study (S1.2)

In [4]:
cmp = pd.DataFrame({n: cooling_agreement(s12.cooling_kw, r["ts"].cooling_kw, STEP_S)
                    for n, r in models.items()}).T
cmp.loc["Edge study"] = cooling_agreement(s12.cooling_kw, s12.cooling_kw, STEP_S)
cmp["Mean cooling / IT power (%)"] = 100 * cmp["Cooling energy (kWh)"] / energy_kwh(s12.it_kw, STEP_S)
cmp = cmp.loc[["Edge study", *models]]
cmp.to_csv(OUT / "cooling_models.csv")
display(vp.format_table(cmp, 1))

ported = models["Ported edge cooling chain"]["ts"]
parts = pd.DataFrame({
    "Edge study": [energy_kwh(s12[c], STEP_S) for c in ("chiller_kw", "fan_kw", "pump_dc_kw", "pump_evap_kw")],
    "Ported edge cooling chain": [energy_kwh(ported[f"cooling_{c}"], STEP_S)
                                  for c in ("chiller_kw", "fan_kw", "pump_dc_kw", "pump_evap_kw")]},
    index=["Chiller", "CRAH fan", "Data-center pump", "Chiller evaporator pump"])
parts["Max absolute difference per step (kW)"] = [
    (ported[f"cooling_{c}"] - s12[c]).abs().max() for c in ("chiller_kw", "fan_kw", "pump_dc_kw", "pump_evap_kw")]
parts.index.name = "Daily energy per component (kWh)"
assert parts["Max absolute difference per step (kW)"].max() < 1e-6
vp.format_table(parts, 1)

,Cooling energy (kWh),Energy difference to edge study (kWh),Energy difference to edge study (%),Mean absolute difference (kW),Max absolute difference (kW),Peak cooling power (kW),Mean cooling / IT power (%)
Edge study,"22,926.0",0.0,0.0,0.0,0.0,"1,301.5",25.4
Constant overhead (0.285 × IT),"25,675.4","2,749.4",12.0,114.6,118.2,"1,407.6",28.5
Linear fit to edge run,"22,926.0",-0.0,-0.0,4.4,20.9,"1,283.1",25.4
Ported edge cooling chain,"22,926.0",0.0,0.0,0.0,0.0,"1,301.5",25.4


,Edge study,Ported edge cooling chain,Max absolute difference per step (kW)
Daily energy per component (kWh),,,
Chiller,"20,019.8","20,019.8",0.0
CRAH fan,"1,296.9","1,296.9",0.0
Data-center pump,"1,149.5","1,149.5",0.0
Chiller evaporator pump,459.8,459.8,0.0


The linear fit (printed below) uses the edge run's step-start values. E03 fitted time-weighted means instead, which gave
a lower intercept, because the edge chain evaluates cooling from the IT power at the start of each step.

In [5]:
print(f"Linear fit: cooling = {slope:.3f} · IT {intercept:+.1f} kW")
e_start = energy_kwh(ported.cooling_kw, STEP_S)
e_mean = energy_kwh(chain_on_mean_it["ts"].cooling_kw, STEP_S)
print(f"Ported chain fed with IT at step start: {e_start:,.1f} kWh; with the 15-min mean IT: {e_mean:,.1f} kWh "
      f"({100 * (e_mean / e_start - 1):+.2f} %)")

Linear fit: cooling = 0.277 · IT -83.0 kW
Ported chain fed with IT at step start: 22,926.0 kWh; with the 15-min mean IT: 23,027.6 kWh (+0.44 %)


In [6]:
fig = vp.stacked_panels([
    ("IT power, step start (kW)", {"IT power": s12.it_kw}),
    ("Cooling electrical power (kW)", {"Edge study": s12.cooling_kw,
                                       **{n: r["ts"].cooling_kw for n, r in models.items()}}),
    ("Cooling power minus edge study (kW)", {n: r["ts"].cooling_kw - s12.cooling_kw for n, r in models.items()}),
], title="IT power and cooling electrical power per cooling model (S1.2, 2025-04-01)", row_height=280)
vp.save(fig, OUT / "cooling_models", png=True)
fig.show()

### 3.2 Outdoor-temperature dependence

In [7]:
temps = np.arange(-5, 40.5, 0.5)
it_mean = s12.it_kw.mean()
const = pd.Series(EDGE_COOLING_OVERHEAD * it_mean, index=temps)
chain_kw = pd.Series(chain.power_kw(it_mean, temps)["total_kw"], index=temps)
fig = vp.stacked_panels([
    ("Chiller COP (–)", {"Ported edge cooling chain": pd.Series(chain.cop(temps), index=temps)}),
    (f"Cooling power at {it_mean:,.0f} kW IT (kW)", {"Constant overhead (0.285 × IT)": const,
                                                      "Ported edge cooling chain": chain_kw}),
], title="Chiller COP and cooling power vs outdoor temperature (shaded: S1.2 day)",
   x_title="Outdoor temperature (°C)", shape="linear", row_height=280)
fig.add_vrect(x0=s12.ambient_temp_c.min(), x1=s12.ambient_temp_c.max(), fillcolor="#bbbbbb",
              opacity=0.25, line_width=0)
vp.save(fig, OUT / "outdoor_temperature", png=True)
fig.show()

In [8]:
rows = {}
for dt, rs in shifted.items():
    ts = rs["Ported edge cooling chain"]["ts"]
    rows[f"+{dt} K"] = {
        "Max outdoor temperature (°C)": s12.ambient_temp_c.max() + dt,
        "Cooling energy, ported chain (kWh)": energy_kwh(ts.cooling_kw, STEP_S),
        "Cooling energy, constant overhead (kWh)": energy_kwh(rs["Constant overhead (0.285 × IT)"]["ts"].cooling_kw, STEP_S),
        "Lowest chiller COP": ts.cooling_cop.min(),
        "Peak cooling power, ported chain (kW)": ts.cooling_kw.max(),
    }
sens = pd.DataFrame(rows).T
sens["Ported chain vs +0 K (%)"] = 100 * (sens["Cooling energy, ported chain (kWh)"]
                                         / sens["Cooling energy, ported chain (kWh)"].iloc[0] - 1)
sens.index.name = "Outdoor temperature shift"
sens.to_csv(OUT / "temperature_sensitivity.csv")
vp.format_table(sens, 1)

,Max outdoor temperature (°C),"Cooling energy, ported chain (kWh)","Cooling energy, constant overhead (kWh)",Lowest chiller COP,"Peak cooling power, ported chain (kW)",Ported chain vs +0 K (%)
Outdoor temperature shift,,,,,,
+0 K,13.4,"22,926.0","25,675.4",4.5,"1,301.5",0.0
+5 K,18.4,"22,926.0","25,675.4",4.5,"1,301.5",0.0
+10 K,23.4,"22,926.0","25,675.4",4.5,"1,301.5",0.0
+15 K,28.4,"22,926.0","25,675.4",4.5,"1,301.5",0.0
+20 K,33.4,"23,955.7","25,675.4",3.8,"1,301.5",4.5
+25 K,38.4,"26,422.1","25,675.4",3.2,"1,301.5",15.2


### 3.3 Room temperature: a stateful thermal actor and a cooling curtailment

The room node integrates `C · dT/dt = IT heat − heat removed` between samples. During the event the heat removal is
capped at 50 % of the IT heat unless the room has reached 27 °C; the actor can only check this at its sample instants.
Without an event the room node gives exactly the ported chain's power (the room stays at its set point).

In [9]:
assert np.allclose(room_base["ts"].cooling_kw, ported.cooling_kw)
base15 = ported.cooling_kw
ev = pd.DataFrame({f"{case}, {lbl}": curtailment_kpis(r, base15, price, EVENT, ROOM["t_max_c"])
                   for (case, lbl), r in room.items()}).T
ev.index.name = "Event, step size"
ev.to_csv(OUT / "curtailment.csv")
vp.format_table(ev, 1)

,Max room temperature (°C),Time above the limit (min),Cooling energy saved in the event window (kWh),Extra cooling energy outside the window (kWh),Net change in cooling energy (kWh),"Largest cooling reduction, 15-min mean (kW)","Largest rebound, 15-min mean (kW)",Change in energy cost (€)
"Event, step size",,,,,,,,
"Curtail, 1 min",27.5,22.0,52.6,62.2,9.7,230.3,135.4,0.8
"Curtail, 5 min",29.7,30.0,65.2,75.9,10.7,381.6,136.3,0.6
"Curtail, 15 min",32.6,60.0,128.2,144.3,16.0,633.8,229.9,0.3
"Precool + curtail, 1 min",27.6,19.0,90.4,104.0,13.7,381.6,135.1,0.4
"Precool + curtail, 5 min",29.6,35.0,128.2,144.8,16.6,381.6,174.0,0.4
"Precool + curtail, 15 min",30.6,45.0,128.2,145.1,16.9,633.8,184.1,0.2


In [10]:
win = slice("2025-04-01 03:30", "2025-04-01 07:30")
def panel(case, col):
    out = {"No event": room_base["ts"][col][win]}
    out.update({lbl: room[(case, lbl)]["ts"][col][win] for lbl in ROOM_STEPS_S})
    return out
import plotly.graph_objects as go
fig = vp.stacked_panels([
    ("Room, curtail only (°C)", panel("Curtail", "cooling_room_c")),
    ("Room, precool + curtail (°C)", panel("Precool + curtail", "cooling_room_c")),
    ("Cooling, precool + curtail (kW)", panel("Precool + curtail", "cooling_kw")),
], title="Room air temperature and cooling power, curtailment 05:00–05:30",
   shape="linear", row_height=300)
limit = dict(color="#888888", dash="dot", width=1.5)
for row in (1, 2):
    fig.add_hline(y=ROOM["t_max_c"], line=limit, row=row, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=limit, name="27 °C limit"), row=1, col=1)
fig.update_traces(line_shape="hv", row=3, col=1)
vp.save(fig, OUT / "room_curtailment", png=True)
fig.show()

In [11]:
cap = pd.DataFrame({f"{c} MJ/K": curtailment_kpis(r, base15, price, EVENT, ROOM["t_max_c"])
                    for c, r in room_capacity.items()}).T
cap.insert(0, "Rise rate after a full stop at mean IT (K/min)",
           [it_mean / (c * 1e3) * 60 for c in CAPACITY_SWEEP_MJ_K])
cap.insert(1, "Stored cold for 20→27 °C, as chiller electricity (kWh)",
           [c * 1e3 * 7 / 3600 / chain.cop(s12.ambient_temp_c).mean() for c in CAPACITY_SWEEP_MJ_K])
cap.index.name = "Thermal capacity (precool + curtail, 1-min step)"
vp.format_table(cap, 1)

,Rise rate after a full stop at mean IT (K/min),"Stored cold for 20→27 °C, as chiller electricity (kWh)",Max room temperature (°C),Time above the limit (min),Cooling energy saved in the event window (kWh),Extra cooling energy outside the window (kWh),Net change in cooling energy (kWh),"Largest cooling reduction, 15-min mean (kW)","Largest rebound, 15-min mean (kW)",Change in energy cost (€)
"Thermal capacity (precool + curtail, 1-min step)",,,,,,,,,,
60 MJ/K,3.8,25.9,29.3,26.0,30.1,36.8,6.7,155.7,44.0,0.2
240 MJ/K,0.9,103.7,27.6,19.0,90.4,104.0,13.7,381.6,135.1,0.4
960 MJ/K,0.2,414.8,27.1,9.0,266.8,293.8,27.0,633.8,231.6,-2.8


### 3.4 Waste-heat recovery (S2.2)

In [12]:
edge_kpis = heat_kpis(s22.heat_pump_kw, s22.heat_pump_kw * s22.heat_pump_cop, s22.heat_to_network_kw,
                      s22.heat_demand_kw, s22.chiller_kw, s22.cooling_kw)
tab = {"Edge study": edge_kpis}
for n, r in heat.items():
    ts = r["ts"]
    tab[n] = heat_kpis(ts.heat_pump_kw, ts.heat_pump_heat_delivered_kw, ts.heat_pump_heat_delivered_kw,
                       ts.heat_pump_heat_demand_kw, ts.cooling_chiller_kw, ts.cooling_kw)
    assert abs(r["metrics"]["Heat delivered (kWh)"] - tab[n]["Heat delivered to the network (kWh)"]) < 1e-6
heat_tab = pd.DataFrame(tab).T
heat_tab.to_csv(OUT / "heat_recovery.csv")
display(vp.format_table(heat_tab, 1))
it_heat = energy_kwh(s22.heat_removed_kw, STEP_S)
cond = edge_kpis["Heat-pump condenser heat (kWh)"]
evap = cond - edge_kpis["Heat-pump electricity (kWh)"]
print(f"Edge S2.2 heat flows over the day (kWh): IT heat removed by the CRAH {it_heat:,.0f}; "
      f"heat-pump evaporator (condenser − electricity) {evap:,.0f}; condenser {cond:,.0f}; "
      f"district-heating exchanger output {energy_kwh(s22.dh_hx_heat_kw, STEP_S):,.0f}; "
      f"accepted by the network {edge_kpis['Heat delivered to the network (kWh)']:,.0f}; "
      f"unmet demand {energy_kwh(s22.heat_demand_kw - s22.heat_to_network_kw, STEP_S):,.0f}")
print("Heat-demand model ported from heating_network.py, max difference (kW):",
      f"{np.abs(heat_demand_kw(s22.ambient_temp_c) - s22.heat_demand_kw).max():.2e}")

,Heat-pump electricity (kWh),Heat-pump condenser heat (kWh),Heat delivered to the network (kWh),Heat demand met (%),Chiller electricity (kWh),Cooling electricity incl. chiller (kWh),Cooling + heat-pump electricity (kWh),Heat revenue (€)
Edge study,"8,758.8","52,011.9","32,423.3",65.2,"8,971.6","10,699.5","19,458.4","2,593.9"
Vessim heat-recovery actor,"5,537.3","33,702.2","33,702.2",67.8,"13,762.2","15,548.2","21,085.5","2,696.2"
"Vessim actor, no exchanger limit","7,847.5","47,763.0","47,763.0",96.0,"11,151.0","12,916.9","20,764.4","3,821.0"


Edge S2.2 heat flows over the day (kWh): IT heat removed by the CRAH 85,886; heat-pump evaporator (condenser − electricity) 43,253; condenser 52,012; district-heating exchanger output 33,615; accepted by the network 32,423; unmet demand 17,306
Heat-demand model ported from heating_network.py, max difference (kW): 1.48e-04


In [13]:
v = heat["Vessim heat-recovery actor"]["ts"]
fig = vp.stacked_panels([
    ("Heat to district heating (kW)", {"District-heating demand": s22.heat_demand_kw,
                                       "Edge study": s22.heat_to_network_kw,
                                       "Vessim heat-recovery actor": v.heat_pump_heat_delivered_kw}),
    ("Heat-pump electrical power (kW)", {"Edge study": s22.heat_pump_kw,
                                         "Vessim heat-recovery actor": v.heat_pump_kw}),
    ("Chiller electrical power (kW)", {"Edge study": s22.chiller_kw,
                                       "Vessim heat-recovery actor": v.cooling_chiller_kw}),
], title="Waste-heat recovery to a warm district-heating network (S2.2, 2025-04-04)", row_height=270)
vp.save(fig, OUT / "heat_recovery", png=True)
fig.show()

### 3.5 Custom code needed

In [14]:
pd.Series({
    "Cooling chain equations (CoolingChain)": logic_lines(CoolingChain),
    "IT actor that exposes its power (PowerProbe)": logic_lines(PowerProbe),
    "Cooling-chain actor (ChainCooling)": logic_lines(ChainCooling),
    "Room thermal node (RoomThermal + Window)": logic_lines(RoomThermal, Window),
    "Heat recovery (HeatRecovery + HeatPumpParams)": logic_lines(HeatRecovery, HeatPumpParams),
    "Heat metric (DeliveredHeat)": logic_lines(DeliveredHeat),
    "District-heating demand (heat_demand_kw)": logic_lines(heat_demand_kw),
}, name="Lines of custom code (no comments or docstrings)").to_frame()

,Lines of custom code (no comments or docstrings)
Cooling chain equations (CoolingChain),45
IT actor that exposes its power (PowerProbe),8
Cooling-chain actor (ChainCooling),13
Room thermal node (RoomThermal + Window),38
Heat recovery (HeatRecovery + HeatPumpParams),31
Heat metric (DeliveredHeat),10
District-heating demand (heat_demand_kw),7


### 3.6 Edge study thermal components and how they map to Vessim

| Edge study component (`simulators/`) | In Vessim | Fidelity |
|---|---|---|
| CRAH, trim chiller, DC pump, evaporator pump (`cooling_crah.py`, `cooling_chiller.py`, `pump_*.py`) | custom actor (algebraic) | exact in heat mode `none` (3.1) |
| Outdoor temperature (`weather.py`) | trace held inside the actor; actors cannot read grid signals | exact |
| Time-shifted chilled-water loop (`water_temp_state.py`) | actor state | not needed in mode `none` (supply stays at set point) |
| Room air | not modelled by the edge study (constant 30 °C CRAH return air); possible as actor state (3.3) | — |
| Stratified hot-water tank, tank controller, thermal switch (`hot_water_tank`, `hot_tank_controller.py`, `thermal_switch.py`) | would be a custom actor with state; Vessim's `Storage` is electrical only and the dispatch policy cannot see it | not ported |
| Heat pump, district-heating HX, heating network (`heat_pump.py`, `district_heating_hx.py`, `heating_network.py`) | custom actor; heat only as state, valued by a custom `Metric` | steady state (3.4) |
| Free-cooling HX, cooling tower, network pump (cold mode) | custom actors | not tested |
| Heat flows between components | Python references between actors, valid only because actors are sampled in list order | — |
| Heat balance / conservation check | none (electrical balance only) | — |

## 4. Evaluation

- **H2 confirmed: Vessim has no thermal model.** Its source has no heat, temperature or cooling term, and actors
  cannot read grid signals, so a temperature-dependent actor holds its own outdoor-temperature trace. Every thermal
  quantity here is user code (≈ 150 lines); Vessim only balances the electrical power the actors return.
- **The edge cooling chain of S1.2 ports exactly.** In heat mode `none` the chain is algebraic: no room-air model
  (constant 30 °C CRAH return air), chilled water held at 15 °C. A 66-line actor (chain equations, IT probe, actor)
  reproduces cooling power and all four components to < 10⁻⁶ kW per step.
- **The edge study's own planning constant is 12 % off its simulated chain.** The controller and scheduler assume
  cooling = 0.285 × IT; the simulated chain gives 25.4 % on this day (+2 749 kWh, +106 kW at the peak). A linear fit
  (0.277 · IT − 83 kW) is within 4.4 kW on average.
- **Outdoor temperature matters only above ≈ 28 °C.** The chiller COP is capped at 4.5 until the Carnot-based value
  falls below it (outdoor ≈ 28.4 °C). The S1.2 day (1.7–13.4 °C) lies in the flat part. Shifting the day by +20 K
  (+25 K) raises cooling energy by 4.5 % (15.2 %). The constant overhead ignores this.
- **Sampling.** The edge chain uses IT power at the start of each 15-min step. Using the step mean gives 0.44 % more
  cooling energy — the same kind of undercount as E01's 0.34 % on IT energy.
- **A stateful thermal actor fits Vessim's interface, but only reacts at samples.** `sample()` is called once per
  step and may advance the model. With a 50 % cooling curtailment (240 MJ/K, ≈ 0.9 K/min), the 27 °C limit is
  overshot to 27.5 °C at a 1-min step, 29.7 °C at 5 min and 32.6 °C at 15 min. Vessim has no events or
  zero-crossing detection.
- **Room thermal mass is a small, short flexibility.** A 30-min curtailment saves 53–128 kWh in the window. The
  rebound uses 62–144 kWh, so the net is +10 to +17 kWh (the fan power is convex) and the cost changes by
  −2.8 to +0.8 €. Precooling to 20 °C reached only ≈ 22 °C, because the CRAH is near its 5 MW design capacity at
  the morning IT peak. Stored cold for 20 → 27 °C is worth 26–415 kWh of chiller electricity for 60–960 MJ/K,
  against a 500 kWh battery.
- **Heat recovery: heat exists only as actor state.** A heat-pump actor (31 lines) plus a 10-line custom `Metric`
  values delivered heat. There is no heat balance, no heat bus and no thermal storage the dispatch policy could use.
  Against the edge S2.2 run, the steady-state actor delivers 3.9 % more heat. It needs 37 % less heat-pump
  electricity, 53 % more chiller electricity and 8.4 % more electricity overall.
- **Two edge study observations to check (S2.2):**
  1. The heat pump's condenser produces 52.0 MWh/day, but the district-heating exchanger passes on only 33.6 MWh.
     The exchanger computes the remainder (`Q_finite_W`), but nothing receives it, so 18.4 MWh/day (35 %)
     disappears. Meanwhile the evaporator still extracts 43.3 MWh from the tank, which relieves the chiller.
  2. The exchanger uses its default 28 kg/s heat-pump-side flow: the heat pump's `m_dot_DH_HX_kg_s` is not
     connected. That caps transfer at ≈ 2.12 MW, although `Q_nom_W` was raised to 5 MW, and 35 % of the
     district-heating demand stays unmet. Without that cap the actor meets 96 % of demand: +1 125 € per day of
     heat revenue.
- **Speed.** 28 runs in 0.8 s; 6 ms per simulated day at 15 min, 89 ms at 1 min.